# Universal Imports

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

import datetime as dt

In [ ]:
# Reading in the sqlite engine 
from core.functions import engine

# Grabbing today's date as food upload date
food_upload_date = dt.date.today().strftime("%m-%d-%Y") # Format: '09-06-2026' September 6th, 2026

print(f"Food upload date: {food_upload_date}")

# Creating food_daily & fact_food tables

In [ ]:
# Create food day table
with engine.begin() as conn:
        conn.execute(text("""CREATE TABLE IF NOT EXISTS food_daily (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            completed TEXT,
                            food_date TEXT,
                            upload_date TEXT)"""))
        
# Create fact food table 
with engine.begin() as conn:
        conn.execute(text("""CREATE TABLE IF NOT EXISTS fact_food (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            meal_time TEXT,
                            food_category TEXT, 
                            value INTEGER,
                            food_daily_id INTEGER,
                            FOREIGN KEY (food_daily_id) REFERENCES food_daily(id) )"""))

# Import food_daily table 

In [ ]:
print("\nUploading new daily food data to food_daily table")

# The idea here is to re-import all data every time because any historical changes I make will be in the cronometer app
food = pd.read_csv(r"C:\Users\ryani\Coding Projects\Habit Tracker Website\cronometer\dailysummary.csv")

# Selecting specific columns then filtering by date for new uploads 
# Need drop duplicates because days are stratified by groups and for this data they're all the same
food_daily = food[["Date", "Completed"]].rename(columns={"Date": "food_date", "Completed": "completed"}).drop_duplicates() 

# Adding today's date as a var to df to see when data was uploaded
food_daily["upload_date"] = food_upload_date 

In [ ]:
# Replacing the dataset in sql every time 
food_daily.to_sql(name="food_daily", con=engine, if_exists='delete_rows', index=False)

print(f"Successfully uploaded {len(food_daily)} rows to the food_daily table")

# Import fact_food table

In [ ]:
#def upload_fact_food(food):

# read back in the data we just uploaded
with engine.connect() as connection: 
    # Renaming food_date back to 'date' for simplicity of merging in IDs
    # and id as food_daily_id to match with fact food table
    food_daily_new = pd.read_sql_query( 
        text("""
            SELECT id as food_daily_id, DATE(food_date) as Date 
            FROM food_daily
            WHERE upload_date = :today_date
        """), connection,                     
        params={"today_date": food_upload_date}, # Using the str version of the datetime for filtering
        dtype={"food_daily_id": "Int64"}) # Parse date here takes out need for pd.to_datetime below 

# Dropping the completed column because it's already accounted for in food_daily
food_filt = food.drop(columns=["Completed"])

# Flipping all the nutrition measurements to long format
food_long = pd.melt(food_filt, id_vars=["Date", "Group"], var_name="food_category", value_name="value")

# Creating the final table by merging in ids from food_daily
fact_food = pd.merge(food_long, food_daily_new, how = "left", on="Date").drop(columns=["Date"]).rename(columns={"Group": "meal_time"})

print("fact_food table successfully created")

In [ ]:
fact_food.to_sql(name="fact_food", con=engine, if_exists='delete_rows', index=False)

print(f"Successfully uploaded {len(fact_food)} rows to the fact_food table")